In [53]:
import pandas as pd
import streamlit as st
import matplotlib.pyplot as plt
import numpy as np

csv_customers_dataset = pd.read_csv('./streamlit_resources/customers_dataset.csv')

csv_orders_dataset = pd.read_csv('./streamlit_resources/orders_dataset.csv')

csv_orders_dataset.dtypes

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

# 1. Clientes por estado y ciudad

Representa una clasificación del número de clientes por estado. Crea una tabla en la que se muestren:

- Estado
- Ciudad
- Número de clientes por ciudad

In [54]:
''' Comprobación de nulos para limpiar los datos
csv_customers_dataset['customer_zip_code_prefix'].isna().any()
csv_customers_dataset['customer_city'].isna().any()
csv_customers_dataset['customer_state'].isna().any() 
No se han encontrado nulos en estas columnas'''

customers_by_city_state = csv_customers_dataset.groupby(
    ['customer_state', 'customer_city']
)['customer_id'].nunique().reset_index()

### Tanto la tabla como los gráficos deberán ser dinámicos respecto a la fecha para permitir el análisis temporal de la evolución de clientes.

In [82]:
'''
Para realizar el filtro tomamos la fecha de compra desde orders_dataset, para compararlo con la fecha actual
'''

# Comprobación de nulos en fecha antes del casting
csv_orders_dataset['order_purchase_timestamp'].isna().any()

# Casting a fecha
csv_orders_dataset['order_purchase_timestamp'] = csv_orders_dataset['order_purchase_timestamp'].apply(pd.to_datetime)

csv_orders_dataset_format = csv_orders_dataset['order_purchase_timestamp'].min()

csv_orders_dataset_format

csv_orders_dataset.dtypes

merge_data_to_compare = pd.merge(csv_orders_dataset,csv_customers_dataset, on='customer_id')

delivered_orders = merge_data_to_compare[merge_data_to_compare['order_status'] == 'delivered']

customers_by_city_state = csv_customers_dataset.groupby(
    ['customer_state', 'customer_city']
)['customer_unique_id'].nunique().reset_index()

dfcustomer = customers_by_city_state.sort_values('customer_unique_id',ascending=False)

dfcustomer


,customer_state,customer_city,customer_unique_id
4176,SP,sao paulo,14984
2788,RJ,rio de janeiro,6620
1062,MG,belo horizonte,2672
601,DF,brasilia,2069
2406,PR,curitiba,1465
...,...,...,...
4298,TO,pequizeiro,1
4297,TO,peixe,1
4292,TO,novo jardim,1
4291,TO,novo alegre,1


# 2. Pedidos por ciudad
## A la tabla anterior añade las siguientes columnas:

Número de pedidos
Porcentaje que representan respecto al total de pedidos
Además, representa el ratio de pedidos por cliente, utilizando el tipo de gráfico que consideres más adecuado.

In [83]:
orders_by_city = merge_data_to_compare.groupby(
    ['customer_state', 'customer_city']
)['order_id'].nunique().reset_index()

total_orders = merge_data_to_compare['order_id'].nunique()

orders_by_city['order_percentage'] = (
    orders_by_city['order_id'] / total_orders
) * 100

customers_by_city = merge_data_to_compare.groupby(
    ['customer_state', 'customer_city']
)['customer_id'].nunique().reset_index()

city_stats = pd.merge(
    customers_by_city,
    orders_by_city,
    on=['customer_state', 'customer_city']
)

city_stats.sort_values('customer_id', ascending=False)

,customer_state,customer_city,customer_id,order_id,order_percentage
4176,SP,sao paulo,15540,15540,15.627357
2788,RJ,rio de janeiro,6882,6882,6.920687
1062,MG,belo horizonte,2773,2773,2.788588
601,DF,brasilia,2131,2131,2.142979
2406,PR,curitiba,1521,1521,1.529550
...,...,...,...,...,...
3029,RS,capivari do sul,1,1,0.001006
3028,RS,capitao,1,1,0.001006
4305,TO,silvanopolis,1,1,0.001006
4287,TO,lajeado,1,1,0.001006


# 3. Análisis de retrasos en pedidos
### Calcula y representa:

Número de pedidos que llegan tarde por ciudad
Porcentaje de pedidos retrasados respecto al total de pedidos de la ciudad
Tiempo medio de retraso en días
Además, al representar esta información, el dashboard deberá incluir un autodiagnóstico que indique la razón más probable del problema.